In [ ]:
# open AI API key 등록
import os
OPENAI_API_KEY=""

# 현재 노트북 커널에 환경변수 등록
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [ ]:
# LangSmith API key 등록
LANGSMITH_TRACING="true" # 추적 할지 말지
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=""
LANGSMITH_PROJECT="Langchain0422"

# 현재 노트북 커널에 환경변수 등록
os.environ['LANGSMITH_TRACING'] = LANGSMITH_TRACING
os.environ['LANGSMITH_ENDPOINT'] = LANGSMITH_ENDPOINT
os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY
os.environ['LANGSMITH_PROJECT'] = LANGSMITH_PROJECT

In [4]:
# 필요 패키지 설치
!pip install -U langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


# 프롬프트 작성 원칙
- 모델이 최대한 정확하고 유용한 정보를 제공할 수 있도록 효과적인 프롬프트를 작성하는 것이 매우 중요

- 명확성과 구체성
  - 질문은 명확하고 구체적이어야 함 (모호한 질문은 LLM 모델의 혼란을 초래)
  - 예시:  "주식 시장은 ?" -> "다음 주 주식 시장에 영향을 줄 수 있는 예정된 이벤트들은 무엇일까?"

- 배경 정보 포함
  - 모델이 문맥을 이해할 수 있도록 필요한 배경 정보 제공 (환각 현상(hallucination) 감소 및 관련성 높은 응답 생성)
  - 예시: "2020년 미국 대선의 결과를 바탕으로 현재 정치 상황에 대한 분석을 해주세요"
  
- 간결함
  - 핵심 정보에 초점을 맞추고, 불필요한 정보는 배제 (복잡하면 덜 중요한 부분에 집중하거나 상당한 영향을 받는 문제 발생)
  - 예시: "2021년에 발표된 삼성전자의 ESG 보고서를 요약해주세요."

- 열린 질문 사용
  - 열린 질문을 통해 모델이 자세하고 풍부한 답변을 제공하도록 유도 (예/아니오로 답변하는 질문보다는 많은 답변을 요구하는 질문)
  - 예시: "신재생에너지에 대한 최신 연구 동향은 무엇인가요?"

- 명확한 목표 설정
  - 얻고자 하는 정보나 결과의 유형을 정확하게 정의 (명확한 지침에 따라 응답을 생성)
  - 예시: "AI 윤리에 대한 문제점과 해결 방안을 요약하여 설명해주세요."

- 언어와 문체
  - 대화의 맥락에 적합한 언어와 문체를 선택 (상황에 맞는 표현 선택)
  - 예시: 공식적인 보고서를 요청하는 경우, "XX 보고서에 대한 전문적인 요약을 부탁드립니다."와 같이 정중한 문체를 사용


# 프롬프트 프레임워크(방식) 예시
### 후가츠식 프롬프트
- 참고 : https://brunch.co.kr/@pletalk/158
- 일본 Note사의 CXO 후가츠 타카유키가 개발한 ChatGPT용 프롬프트 작성을 위한 프레임워크
- ChatGPT를 효율적이고 올바르게 활용할 수 있도록 만들어진 작성 방식

- 형식
  - 지침(명령서), 제약조건, 입력문, 출력문으로 구분
  - 지침과 제약조건 : 입력문을 어떻게 처리할 것인지에 대한 내용을 구체적으로 나열
  - 출력문 : ChatGPT의 답변을 출력
  

<pre>
# 지침:
당신은 {역할}입니다.
아래의 제약조건과 입력문을 기반으로, 최상의 결과를 출력하세요.

# 제약조건:
- 문자수는 {문자수}입니다.
- {제약조건들에 관한 텍스트}

# 입력문:
{입력문 텍스트}

# 출력문:
</pre>

### Chain of Thought (CoT)
- 복잡한 문제를 단계적으로 사고하게 만드는 프롬프트 설계 기법
- GPT 같은 언어 모델이 단순히 답을 “예/아니오”나 숫자로 내는 것이 아니라, 논리적 추론 과정을 거쳐 생각의 흐름을 따라가도록 유도하는 것이 핵심

- 구조

<pre>
[문제 또는 질문]
[Chain of Thought 유도 문장 → Step-by-step reasoning]
[최종 답변 또는 결론]
</pre>

- 작성 팁
  - 사고 흐름을 유도하는 가장 일반적인 표현
  - 논리를 말하게 유도
  - 문제를 세부적으로 나누어 생각하게 함
  - 마치 대화를 하듯 생각 흐름을 자연스럽게 표현
  - 사고의 첫 단계부터 유도

- CoT 구조의 프롬프트 템플릿

<pre>
아래의 단계적 질문 흐름을 따라 논리적으로 사고해줘:

Step 1: 문제 정의 → 사용자나 시장이 겪고 있는 문제를 설명해줘.  
Step 2: 제품 연결 → 해당 제품/서비스가 어떤 해결책을 제공하는지 설명해줘.  
Step 3: 공감 요소 → 타깃이 공감할 만한 포인트 3가지 도출해줘.  
Step 4: 콘텐츠 구상 → 영상이나 콘텐츠로 풀어낼 구체적 구성안을 HOOK, 본문, CTA로 나눠줘.  
Step 5: 바이럴 요소 → 콘텐츠가 퍼질 수 있는 대사, 편집, 음악 아이디어를 제안해줘.  
Step 6: 기대 효과 → 사용자 반응 및 마케팅 효과를 정리해줘.
</pre>

- 마케팅/콘텐츠 기획 프롬프트 예시

<pre>
아래의 단계별 질문 흐름을 따라, TikTok 숏폼 영상 기획을 논리적으로 구성해줘.
각 단계마다 간단한 설명과 함께 아이디어를 제시해.

Step 1: 문제 정의
→ MZ세대 직장인이 왜 단백질 보충이 필요하고, 기존 방식에 어떤 불편을 겪고 있는지 분석해줘.

Step 2: 제품 연결
→ 위 문제에 대해 단백질 음료가 어떤 해결책을 제시할 수 있는지 설명해줘.
특히 ‘환경 친화적’이라는 속성이 어떤 가치를 더하는지도 포함해줘.

Step 3: 타깃 공감 포인트 도출
→ 이 제품을 통해 타깃이 공감할 수 있는 포인트 3가지를 도출해줘.
(예: 간편함, 건강, 윤리적 소비 등)

Step 4: 콘텐츠 콘셉트 기획
→ 위에서 도출된 포인트를 바탕으로 1분 TikTok 영상의 콘셉트를 설명해줘.
HOOK (3초), 본문, CTA 구조로 구성하고, 구체적인 장면 예시도 포함해줘.

Step 5: 바이럴 요소 강화
→ 이 영상이 공유되거나 퍼질 수 있도록, 대사·편집·음악 등에서 반영할 수 있는 바이럴 요소를 2~3가지 제안해줘.

Step 6: 기대 효과 예측
→ 이 콘텐츠를 통해 기대할 수 있는 사용자 반응이나 브랜딩 효과를 논리적으로 정리해줘.
</pre>

# 프롬프트 탬플릿 (PromptTemplate) 구성 요소
- PromptTemplate : 단일 문장 또는 입력변수 등의 <font color=red>간단한 명령을 입력</font>하여 단일 문장 또는 <font color=red>간단한 응답을 생성</font>하는 데 사용되는 프롬프트를 구성할 수 있는 문자열 템플릿
  - <font color=red>지시</font> : 언어 모델에게 어떤 작업을 수행하도록 요청하는 구체적인 지시
  - <font color=red>예시</font> : 요청된 작업을 수행하는 방법에 대한 하나 이상의 예시
  - <font color=red>맥락</font> : 특정 작업을 수행하기 위한 추가적인 맥락
  - <font color=red>질문</font> : 어떤 답변을 요구하는 구체적인 질문

- 예시: 제품 리뷰 요약
  - 지시: "아래 제공된 제품 리뷰를 요약해주세요."
  - 예시: "예를 들어, '이 제품은 매우 사용하기 편리하며 배터리 수명이 길다'라는 리뷰는 '사용 편리성과 긴 배터리 수명이 특징'으로 요약할 수 있습니다."
  - 맥락: "리뷰는 스마트워치에 대한 것이며, 사용자 경험에 초점을 맞추고 있습니다."
  - 질문: "이 리뷰를 바탕으로 스마트워치의 주요 장점을 두세 문장으로 요약해주세요."


- 프롬프트 템플릿의 활용
  - 파라미터
    - <font color=red>template</font>: 문자열 템플릿 및 변수 설정 (중괄호 {} 사용)
    - <font color=red>input_variables</font> : PromptTemplate에서 사용되는 변수(중괄호 안에 들어갈 변수)의 이름을 정의하는 리스트

 - <font color=red>from_template()</font> : 문자열 템플릿을 직접 설정하는 함수

In [5]:
from langchain_openai.chat_models import ChatOpenAI # OpneAI 모델을 생성하는 도구
from langchain_core.prompts import PromptTemplate # 프롬프트 템플릿을 생성하는 도구

In [6]:
llm = ChatOpenAI() # 기본모델 생성

In [7]:
# 프롬프트 템플릿 생성 -> 사용자의 질의를 순서대로 쪼개서 답변을 유도하는 템플릿
QA_template = PromptTemplate.from_template('''
  Qusetion : {q}
  Answer : Let's think step by step
''')
QA_template

PromptTemplate(input_variables=['q'], input_types={}, partial_variables={}, template="\n  Qusetion : {q}\n  Answer : Let's think step by step\n")

In [9]:
# 템플릿으로 프롬프트 만들기
final_prompt = QA_template.format(q="who won the FiFA World Cup in the year 1994?")
final_prompt

"\n  Qusetion : who won the FiFA World Cup in the year 1994?\n  Answer : Let's think step by step\n"

In [10]:
# 기본 LLM 호출하기
res = llm.invoke("Who won the FIFA World Cup in the year 1994?")
# 프롬프트 환성 후 LLM 호출하기
res2 = llm.invoke(final_prompt)

In [11]:
print(res.content)
print(res2.content)

Brazil won the FIFA World Cup in 1994.

In the year 1994, the FIFA World Cup was won by Brazil. They defeated Italy in the final match, which was held in the United States. Brazil won the World Cup for the fourth time in their history.


# LangChain Expression Language (LCEL)

- <font color=red>체인을 구성하고 스트리밍, 배치 및 비동기 지원</font>을 기본적으로 제공하는 선언적 방법
- 기본적으로 파이썬 또는 타입스크립트/자바스크립트를 사용한 체인 만들기의 고수준 대안으로 코드를 사용해 구성할 때 사용하는 모든 기존 랭체인 생성자를 그대로 사용해 체인을 만들 수 있음

# | (파이프라인)으로 체인 생성하기 (LCEL)
  - LangChain 0.1.17 이후 버전부터 사용
  - <font color=red>|</font> 기호는 unix 파이프 연산자와 유사하며, <font color=red>서로 다른 구성 요소를 연결하고 한 구성 요소의 출력을 다음 구성 요소의 입력으로 전달</font>

  - prompt | llm | OutputParser와 같이 체인이 연결된 경우
<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/lcel1.png" width=60%>   
</center>

- 표준 인터페이스 함수을 이용하여 딕셔너리 형태로 값들을 전달
   - <font color=red>invoke()</font> : 딕셔너리 입력에 대해 체인을 호출
   - <font color=red>batch()</font> : 입력 목록에 대해 체인을 호출
   - <font color=red>stream()</font> : 응답의 청크를 스트리밍

In [12]:
# 문자열 파서
from langchain_core.output_parsers import StrOutputParser

In [13]:
# 파서생성
strParser = StrOutputParser()

In [14]:
# LCEL 사용하기
QA_chain = QA_template | llm | strParser

In [15]:
QA_chain.invoke("Who won the FIFA World Cup in the year 1994?")

"First, let's identify which country won the FIFA World Cup in 1994. The FIFA World Cup in 1994 was held in the United States. The winner of the 1994 FIFA World Cup was Brazil. Brazil defeated Italy in the final match, which was held at the Rose Bowl in Pasadena, California. This victory marked Brazil's fourth FIFA World Cup title."

#### 번역기
- LCEL을 이용해 번역하는 체인을 만들어보자
- 모델 : model="gpt-4o-mini" 사용하기

In [24]:
# 모델
llm_4o = ChatOpenAI(model="gpt-4o-mini")
# 프롬프트
translate_template = PromptTemplate.from_template('''
  input : {text}
  위 input으로 들어온 글을 영어로 번역해줘.
  출력형식은 번역된 글자만 나오게 해줘
''')
# 출력 파서
strParser = StrOutputParser()
# LCEL 체인 연결
translate_chain = translate_template | llm_4o | strParser
# 실행
translate_chain.invoke("오늘 날씨는 비가 온다")

'The weather today is rainy.'

##### UI 붙이기
- https://www.gradio.app/

In [27]:
# 응답을 처리하는 알고리즘의 사용자정의함수
def translate_response(message, history):
    return translate_chain.invoke(message)

In [28]:
import gradio as gr

gr.ChatInterface(
    fn=translate_response,
).launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://31621f325e9de3a4f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Chat Model 클래스
- 기능: 메시지의 리스트를 입력으로 받고, 하나의 메시지를 반환
- <font color=red>대화형 메시지를 입력으로 사용하고 대화형 메시지를 출력</font>으로 반환하는 특수화된 LLM 모델
- 일반 텍스트를 사용하는 대신 <font color=red>대화의 맥락을 포함한 메시지를 처리</font>하며, 이를 통해 보다 더 <font color=red>자연스러운 대화를 가능</font>하게 함
- (예시) 사용자가 챗봇과 대화하는 상황에서, 사용자의 질문과 이전 대화 내용을 고려하여 적절한 답변을 생성

- Chat Model 인터페이스의 특징
  - 대화형 입력과 출력
    - 대화의 연속성을 고려하여 입력된 메시지 리스트를 기반으로 적절한 응답 메시지를 생성
    - 챗봇, 가상 비서, 고객 지원 시스템 등 대화 기반 서비스에 활용

  - 다양한 모델 제공 업체와의 통합
    - OpenAI, Cohere, Hugging Face 등 다양한 모델 제공 업체와의 통합을 지원
    - 개발자는 여러 소스의 Chat Models를 조합하여 활용

  - 다양한 작동 모드 지원
    - 랭체인은 동기(sync), 비동기(async), 배치(batching), 스트리밍(streaming) 모드에서 모델을 사용할 수 있는 기능을 제공
    - 다양한 애플리케이션 요구사항과 트래픽 패턴에 따라 유연한 대응이 가능

# ChatPromptTemplate
- <font color=red>대화형 상황에서 여러 메시지 입력을 기반으로 단일 메시지 응답을 생성</font>하는 데 사용
- 대화형 모델이나 챗봇 개발에 주로 사용
- 입력은 여러 메시지를 원소로 갖는 리스트로 구성되며, 각 메시지는 <font color=red>역할(role)</font>과 <font color=red>내용(content)</font>으로 구성

- Message 유형
  - <font color=red>SystemMessage</font>: 시스템 기능을 설명
  - <font color=red>HumanMessage</font>: 사용자 질문을 입력
  - <font color=red>AIMessage</font>: AI 모델의 응답을 제공
  - <font color=red>FunctionMessage</font>: 특정 함수 호출의 결과를 표시
  - <font color=red>ToolMessage</font>: 도구 호출의 결과를 표시

- <font color=red>from_messages()</font> : 2개의 튜플 형태 메시지 리스트를 입력 받아, 각 메시지의 역할(type)과 내용(content)을 기반으로 프롬프트를 구성  
  - 사용자의 입력을 프롬프트에 동적으로 삽입하여, 최종적으로 대화형 상황을 반영한 메시지 리스트를 생성
  - 시스템은 기능을 설명하고, 사용자는 관련 질문을 입력

- 메시지에 출력 양식 포함 (System 메세지 사용하기)

In [30]:
from langchain_core.prompts import ChatPromptTemplate
# PromptTemplate : 1회성 요청과 응답을 구성할때 주로 사용
# ChatPromptTemplate : 멀티턴 대화를 기반으로 파이프라인을 구축할때 주로 사용

In [31]:
llm_4o.invoke("나는 떡볶이를 좋아해").content

'떡볶이는 정말 맛있는 음식이죠! 매콤하고 쫄깃한 떡과 다양한 재료들이 어우러져 풍부한 맛을 느낄 수 있어요. 어떤 스타일의 떡볶이를 좋아하시나요? 클래식한 고추장 떡볶이, 간장 떡볶이, 혹은 seafood 떡볶이 같은 다양한 종류가 있죠!'

In [33]:
# 과거 대화내역을 프롬프트에 삽입해서 질의
chat_template = ChatPromptTemplate.from_messages([
    ('human', '나는 떡볶이를 세상에서 가장 좋아해'),
    ('ai', '아 떡볶이를 좋아하시는군요. 치즈떡볶이, 라볶이, 짜장떡볶이 중에 어떤걸 좋아하세요?'),
    ('human', '나는 짜장떡볶이가 제일 좋아'),
    ('human', '{question}')
])

In [34]:
# 체인구성
simple_chain = chat_template | llm_4o | strParser

simple_chain.invoke("내가 좋아하는 음식이 뭐야?")

'당신이 좋아하는 음식은 짜장떡볶이예요! 맛있죠?'

### placeholder 추가하기
- 메모리 부분을 유연하게 만들기

In [36]:
memory = [
    ('human', '나는 떡볶이를 세상에서 가장 좋아해'),
    ('ai', '아 떡볶이를 좋아하시는군요. 치즈떡볶이, 라볶이, 짜장떡볶이 중에 어떤걸 좋아하세요?'),
    ('human', '나는 짜장떡볶이가 제일 좋아'),
]
# 과거 대화내역을 프롬프트에 삽입해서 질의
chat_template = ChatPromptTemplate.from_messages([
    ('placeholder', '{history}'), # 메모리가 삽입되는 구간
    ('human', '{question}')
])

In [40]:
# 체인구성 및 호출
simple_chain2 = chat_template | llm_4o | strParser

# 사용자 질문 입력
msg = input('질문 : ')
# 두개 이상의 입력인자를 호출하는 방법
ai_res = simple_chain2.invoke({"question" : msg,
                               "history" : memory})
#메모리 업데이트
memory.append(('human', msg))
memory.append(('ai', ai_res))

print('답변 : ', ai_res)

질문 : 나는 짜장떡볶이를 먹어도 될까?
답변 :  짜장떡볶이에 알러지가 있다면 먹지 않는 것이 안전합니다. 알러지는 심각한 반응을 일으킬 수 있으므로 주의가 필요합니다. 대신 다른 맛있는 메뉴를 찾아보는 것이 좋겠어요. 어떤 음식을 좋아하시는지에 따라 다른 추천을 드릴 수 있으니 말씀해 주세요!


In [41]:
memory

[('human', '나는 떡볶이를 세상에서 가장 좋아해'),
 ('ai', '아 떡볶이를 좋아하시는군요. 치즈떡볶이, 라볶이, 짜장떡볶이 중에 어떤걸 좋아하세요?'),
 ('human', '나는 짜장떡볶이가 제일 좋아'),
 ('human', '오늘 점심 메뉴 추천해줘'),
 ('ai',
  '짜장떡볶이를 좋아하신다면, 오늘 점심으로 짜장떡볶이를 드시는 건 어떨까요? 아니면, 짜장떡볶이에 잘 어울리는 사이드 메뉴로 튀김이나 군만두를 추가해도 좋을 것 같아요. 한식이 땡기시면 비빔밥이나 불고기 정식도 맛있겠네요! 어떤 음식이 가장 마음에 드세요?'),
 ('human', '근데 사실 나는 짜장 알러지가 있어서 이제는 못먹어'),
 ('ai',
  '아, 그랬군요. 알러지 때문에 짜장떡볶이를 못 드시는 건 아쉽네요. 대신 다른 메뉴를 추천해 드릴게요! 떡볶이 대신 매운 소스 없이 간장 베이스의 떡볶이, 또는 치즈떡볶이 같은 걸 드셔보는 건 어떨까요? 또 다른 한식 메뉴로는 제육볶음이나 김치찌개, 또는 불고기 덮밥도 좋은 선택일 것 같아요. 어떤 게 끌리세요?'),
 ('human', '나는 짜장떡볶이를 먹어도 될까?'),
 ('ai',
  '짜장떡볶이에 알러지가 있다면 먹지 않는 것이 안전합니다. 알러지는 심각한 반응을 일으킬 수 있으므로 주의가 필요합니다. 대신 다른 맛있는 메뉴를 찾아보는 것이 좋겠어요. 어떤 음식을 좋아하시는지에 따라 다른 추천을 드릴 수 있으니 말씀해 주세요!')]

#### UI 붙이기

In [43]:
# 응답을 처리하는 알고리즘의 사용자정의함수
def chat_response(message, history):
  # 두개 이상의 입력인자를 호출하는 방법
  ai_res = simple_chain2.invoke({"question" : message,
                                "history" : memory})
  #메모리 업데이트
  memory.append(('human', msg))
  memory.append(('ai', ai_res))

  return ai_res

In [44]:
import gradio as gr

gr.ChatInterface(
    fn=chat_response,
).launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab02ad99ec7bba1158.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


- batch() : 입력 리스트에 대해 chain을 호출하는 함수

- stream() : 결과메세지를 청크단위로 받는 함수
  

# 데이터를 효과적으로 전달하는 방법
- https://reference.langchain.com/python/langchain_core/runnables/


### RunnablePassthrough
- RunnablePassthrough
  - 체인에서 데이터 흐름을 제어할 때 유용
  - 특히 질문이나 다른 입력이 동적으로 변경될 가능성이 있는 경우에 많이 사용  
- <font color=red>RunnablePassthrough()</font> : 단순히 입력을 받아 그대로 전달  
- <font color=red>RunnablePassthrough.assign(...)</font> : 입력을 받아 assign 함수에 전달된  인수를 추가
  - 입력 값으로 들어온 값의 key/value 쌍과 새롭게 할당된 key/value 쌍을 합침
- <font color=red>RunnableParallel()</font> : 여러 Runnable 인스턴스를 병렬로 실행
- <font color=red>RunnableLambda()</font> : 사용자 정의 함수를 맵핑, 함수를 체인으로 연결할 수 있게 만들어 줌

- RunnablePassthrough()는 runnable 객체이며, runnable 객체는 invoke() 메소드를 사용하여 별도 실행이 가능

- RunnablePassthrough()로 체인 구성

- RunnablePassthrough.assign()
    - 들어오는 데이터에 추가작업을하고 넘길 수 있다

- RunnableParallel()
    - chain을 병렬로 만드는 기능

- chain에 RunnableParallel 적용하기

- RunnableLambda()
    - 체인 중간에 데이터처리 등을 수행하는 요소
    - 커스텀 runnable 요소를 만드는 용도처럼 활용가능

- RunnableBranch
    - 체인 중간에 분기 시켜주는 runnable
    - 시나리오 : 음식주문 체인을 만들어보자(주문,환불,기타)

# 출력 파서 (Ouput Parser)
- 기능 : 출력 포멧 변경, 정보 추출, 결과 정제, 조건부 로직 적용
- 종류
  - <font color=red>PydanticOuputParser()</font> : 구조화된 정보로 변환
  - <font color=red>CommaSeparatedListOutputParser()</font> : CSV로 변환 (,로 구분)  
  - <font color=red>StrOutputParser()</font> : 문자열로 변환
  - <font color=red>StructuredOutputParser()</font> : dict 형식으로 변환
  - <font color=red>JsonOutputParser()</font> : JSON으로 변환
  - <font color=red>DatetimeOutputParser()</font> : 날짜형식 (datetime)으로 변환
  - <font color=red>EnumOutputParser()</font> : Enum 형식으로 변환 (키와 값을 마침표로 구분)       
  - <font color=red>OutputFixingParser()</font> : 출력 파싱 과정에서 발생할 수 있는 오류를 자동으로 수정하는 기능을 제공   








#### JSON

### CSV